# Graded Response Model — EQSQ (Single Scale)

Fits a single-dimensional GRM to all 120 EQSQ items.

In [ ]:
%load_ext autoreload
%autoreload 2

import os
os.environ['JAX_PLATFORMS'] = 'cpu'

import numpy as np
import jax
import jax.numpy as jnp
import matplotlib.pyplot as plt

## 1. Load Data

In [ ]:
from bayesianquilts.data.eqsq import get_data, item_keys, response_cardinality

df, num_people = get_data(polars_out=True)
print(f"Dataset: {num_people} people, {len(item_keys)} items, {response_cardinality} categories")
df.head()

In [ ]:
SUBSAMPLE_N = num_people
sub_df = df
print(f"Using full dataset: N = {SUBSAMPLE_N}")

## 2. Prepare Data

In [ ]:
def make_data_dict(dataframe):
    data = {}
    for col in dataframe.columns:
        arr = dataframe[col].to_numpy().astype(np.float64)
        data[col] = arr
    data['person'] = np.arange(len(dataframe), dtype=np.float64)
    return data

batch = make_data_dict(sub_df)
n_bad = sum(np.sum(np.isnan(batch[k]) | (batch[k] < 0) | (batch[k] >= response_cardinality)) for k in item_keys)
print(f"Bad/missing values: {n_bad}")

BATCH_SIZE = 256
steps_per_epoch = int(np.ceil(SUBSAMPLE_N / BATCH_SIZE))
print(f"N: {SUBSAMPLE_N}, Batch size: {BATCH_SIZE}, Steps per epoch: {steps_per_epoch}")

def data_factory():
    indices = np.arange(SUBSAMPLE_N)
    np.random.shuffle(indices)
    for start in range(0, SUBSAMPLE_N, BATCH_SIZE):
        end = min(start + BATCH_SIZE, SUBSAMPLE_N)
        idx_batch = indices[start:end]
        yield {k: v[idx_batch] for k, v in batch.items()}

## 3. Fit Baseline GRM (Ignorable Missingness)

In [ ]:
from bayesianquilts.irt.grm import GRModel

model_baseline = GRModel(
    item_keys=item_keys,
    num_people=SUBSAMPLE_N,
    dim=1,
    kappa_scale=0.1,
    response_cardinality=response_cardinality,
    dtype=jnp.float64,
)

NUM_EPOCHS = 200
losses_baseline, params_baseline = model_baseline.fit(
    data_factory,
    batch_size=BATCH_SIZE,
    dataset_size=SUBSAMPLE_N,
    num_epochs=NUM_EPOCHS,
    steps_per_epoch=steps_per_epoch,
    learning_rate=2e-4,
    patience=10,
)
print(f"Baseline final loss: {losses_baseline[-1]:.2f}")

In [ ]:
model_baseline.save_to_disk('grm_baseline')

## 4. Fit MICEBayesianLOO Imputation Model

In [ ]:
from bayesianquilts.imputation.mice_loo import MICEBayesianLOO

imputation_df = sub_df.select(item_keys).to_pandas()
imputation_df = imputation_df.replace(-1, float('nan'))
print(f"NaN count: {imputation_df.isna().sum().sum()}")

mice_loo = MICEBayesianLOO(
    random_state=42,
    prior_scale=1.0,
    pathfinder_num_samples=100,
    pathfinder_maxiter=50,
    batch_size=512,
    verbose=True,
)
mice_loo.fit_loo_models(
    X_df=imputation_df,
    fit_zero_predictors=True,
    n_jobs=-1,
    n_top_features=30,
    seed=42,
)

In [ ]:
mice_loo.save('mice_loo_model.yaml')

## 5. Fit GRM with Analytic Rao-Blackwellized Imputation

In [ ]:
model_imputed = GRModel(
    item_keys=item_keys,
    num_people=SUBSAMPLE_N,
    dim=1,
    kappa_scale=0.1,
    response_cardinality=response_cardinality,
    dtype=jnp.float64,
    imputation_model=mice_loo,
)
model_imputed.validate_imputation_model()

losses_imputed, params_imputed = model_imputed.fit(
    data_factory,
    batch_size=BATCH_SIZE,
    dataset_size=SUBSAMPLE_N,
    num_epochs=NUM_EPOCHS,
    steps_per_epoch=steps_per_epoch,
    learning_rate=2e-4,
    patience=10,
)
print(f"Imputed final loss: {losses_imputed[-1]:.2f}")

In [ ]:
model_imputed.save_to_disk('grm_imputed')

## 6. Compare Results

In [ ]:
plt.figure(figsize=(10, 4))
plt.plot(losses_baseline, label='Baseline (zero-fill)', alpha=0.8)
plt.plot(losses_imputed, label='Analytic Rao-Blackwell imputation', alpha=0.8)
plt.xlabel('Epoch')
plt.ylabel('Loss (neg ELBO)')
plt.title('Training Loss: Baseline vs Imputation')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

In [ ]:
def calibrate_manually(model, n_samples=32, seed=42):
    surrogate = model.surrogate_distribution_generator(model.params)
    key = jax.random.PRNGKey(seed)
    samples = surrogate.sample(n_samples, seed=key)
    expectations = {k: jnp.mean(v, axis=0) for k, v in samples.items()}
    model.calibrated_expectations = expectations
    model.surrogate_sample = samples

calibrate_manually(model_baseline, n_samples=32, seed=101)
calibrate_manually(model_imputed, n_samples=32, seed=102)

In [ ]:
disc_base = np.array(model_baseline.calibrated_expectations['discriminations']).flatten()
disc_imp = np.array(model_imputed.calibrated_expectations['discriminations']).flatten()

fig, ax = plt.subplots(figsize=(10, 16))
y_pos = np.arange(len(item_keys))
width = 0.35
ax.barh(y_pos - width/2, disc_base, width, label='Baseline', alpha=0.7)
ax.barh(y_pos + width/2, disc_imp, width, label='Imputed', alpha=0.7)
ax.set_yticks(y_pos)
ax.set_yticklabels(item_keys, fontsize=6)
ax.set_xlabel('Discrimination')
ax.set_title('Item Discriminations')
ax.legend()
ax.invert_yaxis()
plt.tight_layout()
plt.show()

In [ ]:
ab_base = np.array(model_baseline.calibrated_expectations['abilities']).flatten()
ab_imp = np.array(model_imputed.calibrated_expectations['abilities']).flatten()

fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(ab_base, bins=30, alpha=0.5, label='Baseline', edgecolor='black')
ax.hist(ab_imp, bins=30, alpha=0.5, label='Imputed', edgecolor='black')
ax.set_xlabel('Ability')
ax.set_ylabel('Count')
ax.set_title('Ability Distribution')
ax.legend()
plt.tight_layout()
plt.show()

## Summary

This notebook fitted a single-scale Graded Response Model to all 120 EQSQ
items (E1–E60 + S1–S60) with 4 response categories. Two models were compared:

1. **Baseline** — ignorable missingness (zero-fill for missing values)
2. **Imputed** — analytic Rao-Blackwellized imputation via MICEBayesianLOO

The discrimination parameters indicate how well each item differentiates
respondents along the single latent dimension. The ability distributions
show the estimated latent trait values under each approach.